# CALCUALTE GENE-SET BURDEN SCORES 

# GDC SOMATIC MUTATIONS 
This script calculates the rare variant polygenic risk score for gene sets derived from GDC somatic mutations. The gene-level polygenic risk score files are first downloaded from the workspace bucket. Each gene set is then processed in a loop. For each gene in a given set, the script identifies the corresponding gene-level polygenic risk score file and adds its value to a running total. The final sum represents the polygenic risk score for the entire gene set.

In [2]:
# Set bucket environment
# Load packages
import os
import re
import os
import gc
import glob
import pandas as pd 
from biomart import BiomartServer
from functools import reduce

# Set workspace environment variables
os.environ["WORKSPACE_BUCKET"] = "gs://rw-migration-aou-rw-9f879fc4"
os.environ["ARTIFACT_REGISTRY_DOCKER_REPO"] = (
    "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod"
)

# Load the remaining workspace environment settings
%run Setting_Env_Variables.ipynb

Successfully set WORKSPACE_CDR to: wb-silky-artichoke-2408.C2024Q3R9
Found bucket: id=dataproc-staging-aou-rw-9f879fc4, bucketName=dataproc-staging-wb-fast-radish-6415
Found bucket: id=dataproc-temp-aou-rw-9f879fc4, bucketName=dataproc-temp-wb-fast-radish-6415
Found bucket: id=rw-migration-aou-rw-9f879fc4, bucketName=rw-migration-aou-rw-9f879fc4
Found bucket: id=vwb-aou-datasets-controlled-v8r9, bucketName=vwb-aou-datasets-controlled

Variables extracted:
GOOGLE_CLOUD_PROJECT: wb-fast-radish-6415
WORKSPACE_BUCKET: gs://rw-migration-aou-rw-9f879fc4
WORKSPACE_TEMP_BUCKET: NOT FOUND
WORKSPACE_CDR: wb-silky-artichoke-2408.C2024Q3R9

✅ Saved to /home/dataproc/.bashrc
C2024Q3R9 BQ_DATASET
JupyterLab_Spark_cluster_for_AoU_Spark_20261029 DATAPROC_CLUSTER
dataproc-staging-aou-rw-9f879fc4 GCS_BUCKET
dataproc-temp-aou-rw-9f879fc4 GCS_BUCKET
rw-migration-aou-rw-9f879fc4 GCS_BUCKET
vwb-aou-datasets-controlled-v8r9 GCS_BUCKET
GOOGLE_CLOUD_PROJECT = wb-fast-radish-6415
WORKSPACE_BUCKET = gs://rw-migr

In [ ]:
import subprocess
import numpy as np

# Remove genes from list 
remove = pd.read_csv('gs://rw-migration-aou-rw-9f879fc4/data/chip/hotspots/gene_remove.txt', 
                     sep ='\t', 
                     names=['symbol']) 

# Define paths of gene lists 
path = [
       'gs://rw-migration-aou-rw-9f879fc4/data/gene_burden_analysis/gene_sets/somatic_mutations/aml-frequently-mutated-census-genes-protein-coding-vep-high.tsv',
       'gs://rw-migration-aou-rw-9f879fc4/data/gene_burden_analysis/gene_sets/somatic_mutations/aml-frequently-mutated-census-genes-protein-coding-vep-moderate-high.tsv',
       'gs://rw-migration-aou-rw-9f879fc4/data/gene_burden_analysis/gene_sets/somatic_mutations/aml-frequently-mutated-census-genes-protein-coding.tsv',
       'gs://rw-migration-aou-rw-9f879fc4/data/gene_burden_analysis/gene_sets/somatic_mutations/cancer-frequently-mutated-census-genes-protein-coding-vep-high.tsv',
       'gs://rw-migration-aou-rw-9f879fc4/data/gene_burden_analysis/gene_sets/somatic_mutations/cancer-frequently-mutated-census-genes-protein-coding-vep-moderate-high.tsv',
       'gs://rw-migration-aou-rw-9f879fc4/data/gene_burden_analysis/gene_sets/somatic_mutations/cancer-frequently-mutated-census-genes-protein-coding.tsv'
]

sscore_files = [] 

# Loop through each chromosome 
for CHROM in range(1,23):

    WORKSPACE_BUCKET = os.environ["WORKSPACE_BUCKET"]
    BUCKET = f"{WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_sum_scores/gene_burden_sum_scores_chr{CHROM}/"

    result = subprocess.run(
        ["gsutil", "ls", f"{BUCKET}*.sscore"],
        capture_output=True,
        text=True,
        check=True
    )

    sscore_files.append(sorted(result.stdout.strip().splitlines())) 

sscore_files = np.concatenate(sscore_files) 

# 
for p in path: 
    
    name = os.path.basename(p).replace(".tsv", "")
    
    print(f"Processing {name}")
    
    subset = pd.read_csv(p, sep='\t')
    subset = subset[~subset['symbol'].isin(remove['symbol'])]

    files = [
        path for path in sscore_files
        if any(gene_id in path for gene_id in subset["gene_id"])
    ]
    
    master = None  # Initialize before looping over chromosomes
    for f in files:
        print(f) 
         # Merge chromosome master once per chromosome
        if master is None:
            # If first chromosome, assign master as merged file 
            master = pd.read_csv(f, sep="\t")[["#FID", "IID"]]
            master["TOTAL_SCORE1_SUM"] = 0.0  # running sum initialized
        else:
            print(f) 
       

        df = pd.read_csv(f, sep="\t")
        if not (df["NAMED_ALLELE_DOSAGE_SUM"] == df["SCORE1_SUM"]).all():
            continue

        # Add this gene’s score to running total
        master = master.merge(df[["#FID", "IID", "SCORE1_SUM"]],
                              on=["#FID", "IID"], how="outer")

        master["TOTAL_SCORE1_SUM"] += master["SCORE1_SUM"]
        master.drop(columns=["SCORE1_SUM"], inplace=True)


    # Print shape of merge information 
    print("All chromosomes merged.")
    print("Final shape:", master.shape)
          
    # Save the file 
    local_path = f"{name}-remove-chip-PRS.tsv"
    master.to_csv(local_path, sep ='\t', index=False)
          
    gcs_path = f'{WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_gene_sets/somatic_mutations/'
          
    result = subprocess.run(
        ["gsutil", "cp", local_path, gcs_path],
        capture_output=True,
        text=True,
        check=True
    )
    
    print(f"Finished {name}")
    
    del master  
    
    

In [5]:

import os
import subprocess
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# 1. Remove CHIP hotspot genes
# ---------------------------------------------------------

remove = pd.read_csv(
    "gs://rw-migration-aou-rw-9f879fc4/data/chip/hotspots/gene_remove.txt",
    sep="\t",
    names=["symbol"]
)

# Define paths of gene lists
path = [
    "gs://rw-migration-aou-rw-9f879fc4/data/gene_burden_analysis/gene_sets/somatic_mutations/aml-frequently-mutated-census-genes-protein-coding-vep-high.tsv",
    "gs://rw-migration-aou-rw-9f879fc4/data/gene_burden_analysis/gene_sets/somatic_mutations/aml-frequently-mutated-census-genes-protein-coding-vep-moderate-high.tsv",
    "gs://rw-migration-aou-rw-9f879fc4/data/gene_burden_analysis/gene_sets/somatic_mutations/aml-frequently-mutated-census-genes-protein-coding.tsv",
    "gs://rw-migration-aou-rw-9f879fc4/data/gene_burden_analysis/gene_sets/somatic_mutations/cancer-frequently-mutated-census-genes-protein-coding-vep-high.tsv",
    "gs://rw-migration-aou-rw-9f879fc4/data/gene_burden_analysis/gene_sets/somatic_mutations/cancer-frequently-mutated-census-genes-protein-coding-vep-moderate-high.tsv",
    "gs://rw-migration-aou-rw-9f879fc4/data/gene_burden_analysis/gene_sets/somatic_mutations/cancer-frequently-mutated-census-genes-protein-coding.tsv"
]

WORKSPACE_BUCKET = os.environ["WORKSPACE_BUCKET"]

# ---------------------------------------------------------
# 2. Retrieve SCORE files across all chromosomes
# ---------------------------------------------------------

sscore_files = []

for CHROM in range(1, 23):

    BUCKET = (
        f"{WORKSPACE_BUCKET}/data/gene_burden_analysis/"
        f"burden_sum_scores/gene_burden_sum_scores_chr{CHROM}/"
    )

    result = subprocess.run(
        ["gsutil", "ls", f"{BUCKET}*.sscore"],
        capture_output=True,
        text=True,
        check=True
    )

    sscore_files.extend(
        sorted(result.stdout.strip().splitlines())
    )

sscore_files = np.array(sscore_files)

print(f"Total SCORE files found: {len(sscore_files)}")

# ---------------------------------------------------------
# 3. Process each gene set
# ---------------------------------------------------------

for p in path:

    name = os.path.basename(p).replace(".tsv", "")

    print(f"\nProcessing {name}")

    # Read gene list
    subset = pd.read_csv(p, sep="\t")

    # Remove CHIP hotspot genes
    subset = subset[
        ~subset["symbol"].isin(remove["symbol"])
    ].copy()

    print(f"Genes retained: {len(subset)}")

    # Identify corresponding SCORE files
    files = [
        f for f in sscore_files
        if any(gene_id in f for gene_id in subset["gene_id"])
    ]

    print(f"SCORE files identified: {len(files)}")

    if len(files) == 0:
        print(f"No SCORE files found for {name}. Skipping.")
        continue

    # -----------------------------------------------------
    # 4. Calculate aggregate gene burden scores
    # -----------------------------------------------------

    master = None
    skipped = 0
    processed = 0

    for f in files:

        print(f"Processing: {f}")

        df = pd.read_csv(f, sep="\t")

        # Validate allele dosage
        if not (
            df["NAMED_ALLELE_DOSAGE_SUM"]
            == df["SCORE1_SUM"]
        ).all():

            print("Skipping: NAMED_ALLELE_DOSAGE_SUM mismatch")
            skipped += 1
            continue

        # Retain sample IDs and gene score
        df = df[["#FID", "IID", "SCORE1_SUM"]].copy()

        # Initialize master with first valid gene
        if master is None:

            master = df.rename(
                columns={"SCORE1_SUM": "TOTAL_SCORE1_SUM"}
            )

        else:

            # Merge and sum gene-level scores
            master = master.merge(
                df,
                on=["#FID", "IID"],
                how="outer"
            )

            master["TOTAL_SCORE1_SUM"] = (
                master["TOTAL_SCORE1_SUM"].fillna(0)
                + master["SCORE1_SUM"].fillna(0)
            )

            master.drop(
                columns=["SCORE1_SUM"],
                inplace=True
            )

        processed += 1

    # -----------------------------------------------------
    # 5. Save aggregate scores
    # -----------------------------------------------------

    if master is None:
        print(f"No valid SCORE files for {name}. Skipping.")
        continue

    master["TOTAL_SCORE1_SUM"] = (
        master["TOTAL_SCORE1_SUM"].fillna(0)
    )

    print("\nAll genes processed.")
    print(f"Genes processed: {processed}")
    print(f"Genes skipped: {skipped}")
    print(f"Final shape: {master.shape}")
    
    print(master['TOTAL_SCORE1_SUM'].min())
    print(master['TOTAL_SCORE1_SUM'].max())

    # Save locally
    local_path = f"{name}-remove-chip-PRS.tsv"

    master.to_csv(
        local_path,
        sep="\t",
        index=False
    )

    # Copy results to GCS
    gcs_path = (
        f"{WORKSPACE_BUCKET}/data/gene_burden_analysis/"
        "burden_gene_sets/somatic_mutations/"
    )

    subprocess.run(
        ["gsutil", "cp", local_path, gcs_path],
        capture_output=True,
        text=True,
        check=True
    )

    print(f"Finished {name}")
    print(f"Uploaded: {gcs_path}{local_path}")

    del master


Total SCORE files found: 18078

Processing aml-frequently-mutated-census-genes-protein-coding-vep-high
Genes retained: 71
SCORE files identified: 62
Processing: gs://rw-migration-aou-rw-9f879fc4/data/gene_burden_analysis/burden_sum_scores/gene_burden_sum_scores_chr1/EUR_HC-burden-prs-ENSG00000054118-chr1.sscore
Processing: gs://rw-migration-aou-rw-9f879fc4/data/gene_burden_analysis/burden_sum_scores/gene_burden_sum_scores_chr1/EUR_HC-burden-prs-ENSG00000065526-chr1.sscore
Processing: gs://rw-migration-aou-rw-9f879fc4/data/gene_burden_analysis/burden_sum_scores/gene_burden_sum_scores_chr1/EUR_HC-burden-prs-ENSG00000117713-chr1.sscore
Processing: gs://rw-migration-aou-rw-9f879fc4/data/gene_burden_analysis/burden_sum_scores/gene_burden_sum_scores_chr1/EUR_HC-burden-prs-ENSG00000119535-chr1.sscore
Processing: gs://rw-migration-aou-rw-9f879fc4/data/gene_burden_analysis/burden_sum_scores/gene_burden_sum_scores_chr2/EUR_HC-burden-prs-ENSG00000066032-chr2.sscore
Processing: gs://rw-migration-a